# Notebook to experiment and create LED tracker. Stable version in scripts/LED_times.py

This notebook is mainly for experiments. Goes from initial scripts to test on images, to working code for videos (which is the final implementation) at the end.

In [3]:
from skimage import measure, color
import numpy as np
import argparse
import imutils
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

## images

In [ ]:
led_img= "C:\\Users\\arite\\Desktop\\ASM RA\\ASM_Killifish_repo\\d48_65_s1.png"
no_led_img = "/Users/asmlabuser1/Scripts_Ari/no_led.png"

In [ ]:
led = cv2.cvtColor(cv2.imread(led_img), cv2.COLOR_BGR2RGB)
# no_led = cv2.cvtColor(cv2.imread(no_led_img), cv2.COLOR_BGR2RGB)
plt.imshow(led)

In [ ]:
print(led[0][0]) #pixel of normal image
led_LAB = color.rgb2lab(led)
# led_LAB = led
print(led_LAB[0][0])

In [ ]:
pixels = []
for i,y in enumerate(led_LAB):
    for j, x in enumerate(y):
        if x[0] > 35 and x[1] > 25 and -20<x[2]<40:
            pixels.append([i,j])
pixels_np = np.asarray(pixels)
print(pixels_np)

In [ ]:
mask = np.zeros((len(led), len(led[0])))
for [i,j] in pixels:
    mask[i][j] = 255
plt.imshow(mask)

In [ ]:
mask2 = cv2.dilate(mask2, None, iterations=1)
mask2 = cv2.erode(mask, None, iterations=3)

plt.imshow(mask2)

In [ ]:
print(no_led[0][0]) #pixel of normal image
no_led_LAB = color.rgb2lab(no_led)
print(no_led_LAB[0][0])

In [ ]:
pixels_no = []
for i,y in enumerate(no_led_LAB):
    for j, x in enumerate(y):
        if x[0] > 35 and x[1] > 25 and -20<x[2]<40:
            pixels_no.append([i,j])
# pixels_np_no = np.asarray(pixels_no)
# print(pixels_np)

In [ ]:
mask_no = np.zeros((1000,1600))
for [i,j] in pixels_no:
    mask_no[i][j] = 255
plt.imshow(mask_no)

In [ ]:
mask_no2 = cv2.erode(mask_no, None, iterations=3)
mask_no2 = cv2.dilate(mask_no2, None, iterations=8)
plt.imshow(mask_no2)

In [ ]:
led_img= "C:\\Users\\arite\\Desktop\\ASM RA\\ASM_Killifish_repo\\f1_noled.png"

led_LAB = cv2.cvtColor(cv2.imread(led_img), cv2.COLOR_BGR2LAB)
# led_LAB = color.rgb2lab(led)
plt.imshow(led_LAB)
and_mask = np.logical_and(led_LAB[:, :, 0] > 25, led_LAB[:, :, 1] > 25)
and_mask = np.logical_and(and_mask, np.logical_and(led_LAB[:, :, 2] > -20, led_LAB[:, :, 2] < 40))
pixels = np.argwhere(and_mask)

mask_bg = np.zeros((len(led_LAB), len(led_LAB[0])))
mask_bg[pixels[:, 0], pixels[:, 1]] = 255

# mask2 = cv2.dilate(mask_bw, None, iterations=3)
# mask_bg = cv2.erode(mask2, None, iterations=5)

plt.imshow(mask_bg)

In [ ]:
# %%timeit

led_img= "C:\\Users\\arite\\Desktop\\ASM RA\\ASM_Killifish_repo\\f1_led.png"
no_led_img = "C:\\Users\\arite\\Desktop\\ASM RA\\ASM_Killifish_repo\\f1_noled.png"
# led_LAB = cv2.cvtColor(cv2.imread(led_img), cv2.COLOR_BGR2Lab)
# led_LAB = cv2.cvtColor(cv2.imread(led_img), cv2.COLOR_BGR2Lab)

# plt.imshow(led_LAB)
# print(led_LAB[800][500])

# print(led_LAB[500][800])
# led_LAB = color.rgb2lab(led)

# and_mask = np.logical_and(led_LAB[:, :, 0] > 25, led_LAB[:, :, 1] > 25)
# and_mask = np.logical_and(and_mask, np.logical_and(led_LAB[:, :, 2] > -20, led_LAB[:, :, 2] < 40))

# pixels = np.argwhere(and_mask)
# lower_threshold = (25*2.55, 25+128, -20+128)
# upper_threshold = (100*2.55, 128+128, 40+128)
# mask = cv2.inRange(led_LAB, lower_threshold, upper_threshold)
# plt.imshow(mask)
led = create_mask(cv2.imread(led_img))
noled = create_mask(cv2.imread(no_led_img))
plt.imshow(led-noled)
# maskcv =  
# mask_bw = np.zeros((len(led_LAB), len(led_LAB[0])))
# mask_bw[pixels[:, 0], pixels[:, 1]] = 255
# mask_without_bg = mask_bw-mask_bg
# mask2 = cv2.dilate(mask_without_bg, None, iterations=3)
# mask_erode = cv2.erode(mask2, None, iterations=5)
print(np.count_nonzero(mask))
# plt.imshow(mask_erode)
# plt.imshow(mask_bw)

Work on videos + is some sort of verification needed??

## video
this section contains the current final implementation. we use video path to find all video files. LED_times array saves the final result for each video before saving that data in a csv. 

create_mask creates a mask given the image/frame. it converts frame to LAB color format and applies a small gaussian blur to decrease noise, before applying a threshold on L for luminance(brightness), and the other 2 channels for filtering the red color. 

the for loop is the main code. for each video, a progress bar is initiated. a capture object is opened to process each frame using opencv. bg flag tracks if a background mask has been created or not, which is done on the first frame. the small bg if loop stores the background mask in bg_mask variable and then turns flag to false. Next, we go through all the frames wherein a mask for the frame is created, after which we subtract background mask from the current frame's mask. we store all the timestamps at which LED is observed, and append the max and the min of this along with the video name to LED_times. After all the videos are processed, this array is saved to a csv in data/output folder

--vis flag is used to visualise background mask, and current mask while LED is detected. 

--video_path can be used to provide path to a folder containing videos if not in data/videos. 

--output_path can be used to provide output path for csv if not data/output






In [55]:
import pandas as pd
vid_path=Path("./data/videos/")
# capture = cv2.VideoCapture(vid_path)
vid_list = list(vid_path.glob("*.mp4"))
# video = cv2.VideoCapture("d48_65_S1.mp4")
print(vid_list)
LED_times = [["name", "start time(s)", "end time(s)"]]
print(LED_times)

out_path = Path("./data/output/")
vis = False

[PosixPath('data/videos/d48_35_F1.mp4'), PosixPath('data/videos/d48_23_T3.mp4'), PosixPath('data/videos/d48_65_S1.mp4'), PosixPath('data/videos/d48_61_T1.mp4')]
[['name', 'start time(s)', 'end time(s)']]


In [82]:
def create_mask(frame):
  frame_LAB = cv2.cvtColor(cv2.GaussianBlur(frame,(5,5),0), cv2.COLOR_BGR2Lab)
  lower_threshold = (27*2.55, 25+128, -20+128)
  upper_threshold = (100*2.55, 128+128, 40+128)
  mask = cv2.inRange(frame_LAB, lower_threshold, upper_threshold)
  return mask

for video_path in vid_list[1:2]:
  video = cv2.VideoCapture(str(video_path))
  print(video_path.name)
  

  total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
  progress_bar = tqdm(total=total_frames)
  
  time_for_video=[]
  bg = True
  mask_bg = []
  
  # Read until video is completed
  while(video.isOpened()):
    # Capture frame-by-frame
    ret, frame = video.read()

    if bg == True and ret == True:
      mask_bg = create_mask(frame)
      bg = False
      if vis == True: 
        cv2.imshow(f'{video_path.name} bg',mask_bg)
    if ret == True:  
      # Press Q on keyboard to  exit, for visualization
      if cv2.waitKey(25) & 0xFF == ord('q'):
        break
      
      mask_frame = create_mask(frame)
      mask_bw = mask_frame - mask_bg
      # print(time_for_video)
      #find number of non-zero pixels and print timestamp and pixels count if nonzero pixels > 2500
      if np.count_nonzero(mask_bw) > 2500:


        time_for_video.append(video.get(cv2.CAP_PROP_POS_MSEC))
        print(min(time_for_video)) 
        # print timestamp of video frame
        # print(video.get(cv2.CAP_PROP_POS_MSEC))
        # EXPERIMENT IF ERODE AND DILATE IS NEEDED
        if vis == True:  
          mask_dilate = cv2.dilate(mask_bw, None, iterations=3)
          mask_erode = cv2.erode(mask_dilate, None, iterations=4)
          cv2.imshow(f'{video_path.name} mask',mask_erode)
      progress_bar.update(1)
    # Break the loop
    else: 
      break
    
  LED_times.append([video_path.name, min(time_for_video)/1000, max(time_for_video)/1000])
  # When everything done, release the video capture object, progress bar, and close frames
  
  video.release()
  progress_bar.close()
  cv2.destroyAllWindows()
  

np.savetxt(out_path + "LED_times.csv",LED_times, delimiter=',', fmt = ["%s","%.2f", "%.2f"] )

d48_23_T3.mp4


 50%|████▉     | 2676/5370 [01:39<01:43, 26.13it/s]

89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667


 50%|████▉     | 2682/5370 [01:39<01:43, 26.09it/s]

89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667


...

89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667


 52%|█████▏    | 2819/5370 [01:44<01:33, 27.18it/s]

89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667
89122.36666666667


 61%|██████▏   | 3302/5370 [02:03<01:18, 26.23it/s]

KeyboardInterrupt: 

In [90]:
video.release()
progress_bar.close()
cv2.destroyAllWindows()


In [91]:
print(time_for_video)

[89122.36666666667, 89155.73333333335, 89189.1, 89222.46666666667, 89255.83333333334, 89289.20000000001, 89322.56666666668, 89355.93333333333, 89389.3, 89422.66666666667, 89456.03333333334, 89489.40000000001, 89522.76666666666, 89556.13333333333, 89589.5, 89622.86666666667, 89656.23333333334, 89689.6, 89722.96666666666, 89756.33333333334, 89789.70000000001, 89823.06666666668, 89856.43333333335, 89889.8, 89923.16666666667, 89956.53333333334, 89989.90000000001, 90023.26666666668, 90056.63333333333, 90090.0, 90123.36666666667, 90156.73333333334, 90190.1, 90223.46666666666, 90256.83333333333, 90290.2, 90323.56666666667, 90356.93333333335, 90390.30000000002, 90423.66666666667, 90457.03333333334, 90490.40000000001, 90523.76666666668, 90557.13333333335, 90590.5, 90623.86666666667, 90657.23333333334, 90690.6, 90723.96666666667, 90757.33333333333, 90790.7, 90824.06666666667, 90857.43333333333, 90890.8, 90924.16666666667, 90957.53333333334, 90990.90000000001, 91024.26666666668, 91057.63333333335

In [84]:
print(LED_times)
# print(time_for_video)
LED_times.append([str(video_path.name), "{:.2f}".format(min(time_for_video)/1000), "{:.2f}".format(max(time_for_video)/1000)])

[['name', 'start time(s)', 'end time(s)'], ['d48_23_T3.mp4', '89.12', '94.09'], ['d48_23_T3.mp4', '89.12', '94.09']]


In [85]:
np.savetxt(Path(out_path, "LED_times.csv"),LED_times, delimiter=',', fmt = "%s" )

In [92]:
!python ./scripts/LED_times.py --debug

 Videos to be processed are ['d48_23_T3.mp4']
d48_23_T3.mp4
 50%|███████████████████▍                   | 2671/5370 [01:38<01:34, 28.71it/s]89122.36666666667 89122.36666666667
89122.36666666667 89155.73333333335
89122.36666666667 89189.1
 50%|███████████████████▍                   | 2674/5370 [01:38<01:34, 28.56it/s]89122.36666666667 89222.46666666667
89122.36666666667 89255.83333333334
89122.36666666667 89289.20000000001
 50%|███████████████████▍                   | 2677/5370 [01:38<01:35, 28.23it/s]89122.36666666667 89322.56666666668
89122.36666666667 89355.93333333333
89122.36666666667 89389.3
 50%|███████████████████▍                   | 2680/5370 [01:38<01:35, 28.21it/s]89122.36666666667 89422.66666666667
89122.36666666667 89456.03333333334
89122.36666666667 89489.40000000001
 50%|███████████████████▍                   | 2683/5370 [01:38<01:35, 28.06it/s]89122.36666666667 89522.76666666666
89122.36666666667 89556.13333333333
89122.36666666667 89589.5
 50%|███████████████████▌     